In [1]:
!pip install ultralytics gdown -q

import torch
print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
else:
    print('ATTENTION : Pas de GPU !')
    print('Aller dans Settings → Accelerator → GPU T4 x2')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.6 MB/s eta 0:00:00
CUDA disponible : True
GPU : Tesla T4


In [2]:
import os

DATASET_PATH = '/kaggle/input/datasets/benkhalifamahmoud/bdd100k-highway/bdd100k_highway'

TRAIN_IMG = os.path.join(DATASET_PATH, 'images', 'train')
VAL_IMG   = os.path.join(DATASET_PATH, 'images', 'val')
TRAIN_LBL = os.path.join(DATASET_PATH, 'labels', 'train')
VAL_LBL   = os.path.join(DATASET_PATH, 'labels', 'val')

n_train_img = len([f for f in os.listdir(TRAIN_IMG) if f.endswith('.jpg')])
n_val_img   = len([f for f in os.listdir(VAL_IMG)   if f.endswith('.jpg')])
n_train_lbl = len([f for f in os.listdir(TRAIN_LBL) if f.endswith('.json')])
n_val_lbl   = len([f for f in os.listdir(VAL_LBL)   if f.endswith('.json')])

print(f'Train : {n_train_img} images | {n_train_lbl} labels')
print(f'Val   : {n_val_img} images | {n_val_lbl} labels')

Train : 17414 images | 17414 labels
Val   : 2499 images | 2499 labels


In [3]:
import json
import shutil

CLASS_MAP = {
    'car':           0,
    'truck':         1,
    'bus':           2,
    'pedestrian':    3,
    'rider':         4,
    'traffic sign':  5,
    'traffic light': 6
}

IMG_W, IMG_H = 1280, 720

OUT_TRAIN_LBL = '/kaggle/working/dataset/labels/train'
OUT_VAL_LBL   = '/kaggle/working/dataset/labels/val'
OUT_TRAIN_IMG = '/kaggle/working/dataset/images/train'
OUT_VAL_IMG   = '/kaggle/working/dataset/images/val'

for folder in [OUT_TRAIN_LBL, OUT_VAL_LBL, OUT_TRAIN_IMG, OUT_VAL_IMG]:
    os.makedirs(folder, exist_ok=True)


def convert_split(lbl_dir, img_src_dir, out_lbl_dir, out_img_dir, split):
    json_files = [f for f in os.listdir(lbl_dir) if f.endswith('.json')]
    converted, skipped = 0, 0

    print(f'Conversion {split} : {len(json_files)} fichiers...')

    for i, jf in enumerate(json_files):
        with open(os.path.join(lbl_dir, jf), 'r') as f:
            data = json.load(f)

        frames = data.get('frames', [])
        if not frames:
            skipped += 1
            continue
        objects = frames[0].get('objects', [])

        yolo_lines = []
        for obj in objects:
            cat = obj.get('category', '')
            if cat not in CLASS_MAP:
                continue
            box = obj.get('box2d')
            if not box:
                continue

            x1, y1, x2, y2 = box['x1'], box['y1'], box['x2'], box['y2']
            x_center = min(max((x1 + x2) / (2 * IMG_W), 0), 1)
            y_center = min(max((y1 + y2) / (2 * IMG_H), 0), 1)
            width    = min(max((x2 - x1) / IMG_W, 0), 1)
            height   = min(max((y2 - y1) / IMG_H, 0), 1)

            yolo_lines.append(
                f"{CLASS_MAP[cat]} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
            )

        if not yolo_lines:
            skipped += 1
            continue

        base_name = jf.replace('.json', '')
        img_name  = base_name + '.jpg'
        src_img   = os.path.join(img_src_dir, img_name)

        if not os.path.exists(src_img):
            skipped += 1
            continue

        txt_path = os.path.join(out_lbl_dir, base_name + '.txt')
        with open(txt_path, 'w') as f:
            f.write('\n'.join(yolo_lines))

        shutil.copy2(src_img, os.path.join(out_img_dir, img_name))
        converted += 1

        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(json_files)}...')

    print(f'{split.upper()} — Convertis : {converted} | Ignorés : {skipped}')
    return converted


print('=== Conversion TRAIN ===')
n_train = convert_split(TRAIN_LBL, TRAIN_IMG, OUT_TRAIN_LBL, OUT_TRAIN_IMG, 'train')

print('\n=== Conversion VAL ===')
n_val = convert_split(VAL_LBL, VAL_IMG, OUT_VAL_LBL, OUT_VAL_IMG, 'val')

print(f'\nDataset converti : {n_train} train | {n_val} val')

=== Conversion TRAIN ===
Conversion train : 17414 fichiers...
  2000/17414...
  4000/17414...
  6000/17414...
  8000/17414...
  10000/17414...
  12000/17414...
  14000/17414...
  16000/17414...
TRAIN — Convertis : 17413 | Ignorés : 1

=== Conversion VAL ===
Conversion val : 2499 fichiers...
  2000/2499...
VAL — Convertis : 2499 | Ignorés : 0

Dataset converti : 17413 train | 2499 val


In [4]:

yaml_content = """# Dataset BDD100K — Scénario Autoroute
# Filtré sur scene=highway | 7 classes

path: /kaggle/working/dataset
train: images/train
val: images/val

nc: 7

names:
  0: car
  1: truck
  2: bus
  3: pedestrian
  4: rider
  5: traffic sign
  6: traffic light
"""

YAML_PATH = '/kaggle/working/dataset_autoroute.yaml'
with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print('dataset_autoroute.yaml créé')
print(yaml_content)

dataset_autoroute.yaml créé
# Dataset BDD100K — Scénario Autoroute
# Filtré sur scene=highway | 7 classes

path: /kaggle/working/dataset
train: images/train
val: images/val

nc: 7

names:
  0: car
  1: truck
  2: bus
  3: pedestrian
  4: rider
  5: traffic sign
  6: traffic light



In [5]:
from ultralytics import YOLO

print('=== MODÈLE 1 : YOLOv8s — Baseline ===')

model_s = YOLO('yolov8s.pt')
results_s = model_s.train(
    data=YAML_PATH,

    epochs=80,           # Kaggle = temps limité → 80 optimal
    imgsz=640,           # 1280 trop lent sur Kaggle
    batch=16,            # T4/P100 → max stable
    device=0,
    name='yolov8s_autoroute',

    optimizer='AdamW',
    lr0=0.001,
    lrf=0.1,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,    
    cos_lr=True,         
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,

    fliplr=0.0,         
    mosaic=0.7,
    mixup=0.1,
    copy_paste=0.1,

    translate=0.1,
    scale=0.5,
    degrees=0.0,

    close_mosaic=10,    

    save_period=10,    
    plots=True,
    verbose=True,

    patience=20,
)

map50_s   = results_s.results_dict.get('metrics/mAP50(B)', 0)
map5095_s = results_s.results_dict.get('metrics/mAP50-95(B)', 0)
prec_s    = results_s.results_dict.get('metrics/precision(B)', 0)
rec_s     = results_s.results_dict.get('metrics/recall(B)', 0)

print(f'mAP@0.5      : {map50_s:.4f}')
print(f'mAP@0.5:0.95 : {map5095_s:.4f}')
print(f'Précision    : {prec_s:.4f}')
print(f'Rappel       : {rec_s:.4f}')
import shutil
shutil.copy(
    '/kaggle/working/runs/detect/yolov8s_autoroute/weights/best.pt',
    '/kaggle/working/best_yolov8s.pt'
)
print('✅ YOLOv8s sauvegardé dans Output !')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
=== MODÈLE 1 : YOLOv8s — Baseline ===
Ultralytics 8.4.34 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset_autoroute.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h

In [6]:
import os
for root, dirs, files in os.walk('/kaggle/working/runs'):
    for f in files:
        if f.endswith('.pt'):
            print(os.path.join(root, f))

/kaggle/working/runs/detect/yolov8s_autoroute/weights/epoch50.pt
/kaggle/working/runs/detect/yolov8s_autoroute/weights/epoch60.pt
/kaggle/working/runs/detect/yolov8s_autoroute/weights/epoch30.pt
/kaggle/working/runs/detect/yolov8s_autoroute/weights/last.pt
/kaggle/working/runs/detect/yolov8s_autoroute/weights/epoch70.pt
/kaggle/working/runs/detect/yolov8s_autoroute/weights/epoch20.pt
/kaggle/working/runs/detect/yolov8s_autoroute/weights/epoch0.pt
/kaggle/working/runs/detect/yolov8s_autoroute/weights/epoch40.pt
/kaggle/working/runs/detect/yolov8s_autoroute/weights/epoch10.pt
/kaggle/working/runs/detect/yolov8s_autoroute/weights/best.pt
